# Module 1 : Création du Master Dataset de Features (Intraday)

**Objectif :** Calculer l'intégralité des features techniques sur tout l'historique disponible.
**Sortie :** Un fichier unique `LSTMt_features_15m.parquet` contenant toutes les données (Train + Val + Test) sous-échantillonnées à 15 minutes, sans normalisation ni découpage.

In [1]:
import pandas as pd
import numpy as np
import pandas_ta as ta
from pathlib import Path
import gc
import warnings
import datetime as dt

warnings.filterwarnings('ignore')

# --- Configuration ---
DATA_FILE_PATH = Path("../Raw Data/raw_data_set.parquet")
OUTPUT_DIR = Path("./parquets")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = ['EURUSD', 'GBPUSD', 'USDCAD', 'USDJPY', 'AUDUSD', 'NZDUSD']
N_PAIRS = len(PAIRS)
H = 240 # Horizon cible (minutes)
EPSILON = 1e-9
PREDICTION_FREQUENCY = 15 # Sous-échantillonnage final

print("Configuration chargée.")

Configuration chargée.


In [2]:
print("--- Chargement des données brutes ---")
if not DATA_FILE_PATH.exists():
    raise FileNotFoundError(f"ERREUR : Fichier {DATA_FILE_PATH} introuvable.")

FX_df_raw = pd.read_parquet(DATA_FILE_PATH).sort_index()

# Nettoyage initial des bornes temporelles (minutes pleines)
start_date = FX_df_raw.index[0].ceil('h')
end_date_exclusive = FX_df_raw.index[-1].floor('h')
end_date = end_date_exclusive - pd.Timedelta(minutes=1)
FX_df = FX_df_raw.loc[start_date:end_date].copy()

del FX_df_raw
gc.collect()
print(f"Données chargées : {FX_df.shape}")

--- Chargement des données brutes ---
Données chargées : (10873200, 24)


In [3]:
print("--- Calcul des Features (Groupes 0-5) ---")
all_features_df = pd.DataFrame(index=FX_df.index)

# Groupe 0 : Base & Cible
for pair in PAIRS:
    high = np.log(FX_df[f'FX_{pair}_high'])
    low = np.log(FX_df[f'FX_{pair}_low'])
    last = np.log(FX_df[f'FX_{pair}_last'])
    volume = FX_df[f'FX_{pair}_volume']
    
    log_range = high - low
    log_return = last.diff()
    
    # Cible (Volatilité future moyenne sur H minutes)
    target = log_range.rolling(window=H, min_periods=H).mean().shift(-(H))
    
    all_features_df[f'{pair}_vol_cible_bps'] = target * 10000
    all_features_df[f'{pair}_log_range'] = log_range
    all_features_df[f'{pair}_log_return'] = log_return

    # Groupe 1 : Agrégations (5m, 15m, 60m)
    for window in [5, 15, 60]:
        all_features_df[f'{pair}_realized_vol_{window}m'] = log_range.rolling(window).sum()
        all_features_df[f'{pair}_momentum_{window}m'] = log_return.rolling(window).sum()

    # Groupe 2 : Structure de Volatilité
    ewm_240 = log_range.ewm(span=240, adjust=False).mean()
    all_features_df[f'{pair}_vol_term_structure'] = log_range.ewm(span=60, adjust=False).mean() / (ewm_240 + EPSILON) - 1
    all_features_df[f'{pair}_vol_momentum_4h'] = ewm_240 / (ewm_240.shift(240) + EPSILON) - 1

    # Groupe 3 : Risque (Kurtosis, Skew, Asymétrie)
    all_features_df[f'{pair}_kurtosis_returns_4h'] = log_return.rolling(240).kurt()
    all_features_df[f'{pair}_skewness_returns_4h'] = log_return.rolling(240).skew()
    
    vol_down = np.sqrt((log_return.where(log_return < 0, 0)**2).rolling(240).mean())
    vol_up = np.sqrt((log_return.where(log_return > 0, 0)**2).rolling(240).mean())
    all_features_df[f'{pair}_vol_asymmetry_ratio_4h'] = vol_down / (vol_up + EPSILON) - 1

    # Groupe 4 : Anomalies
    all_features_df[f'{pair}_return_zscore_60m'] = (log_return - log_return.rolling(60).mean()) / (log_return.rolling(60).std() + EPSILON)
    all_features_df[f'{pair}_prix_momentum_4h'] = np.exp(last) / np.exp(last).shift(240) - 1 # Usage du prix brut reconstitué ou log return sum

    # Groupe 5 : Flux & Volume
    mfi = ta.mfi(high=np.exp(high), low=np.exp(low), close=np.exp(last), volume=volume, length=14)
    all_features_df[f'{pair}_mfi_centered'] = mfi / 50.0 - 1.0
    all_features_df[f'{pair}_volume_zscore_4h'] = (volume - volume.rolling(240).mean()) / (volume.rolling(240).std() + EPSILON)

print("Groupes 0-5 calculés.")

--- Calcul des Features (Groupes 0-5) ---
Groupes 0-5 calculés.


In [4]:
print("--- Calcul des Features Cross-Section (Groupe 6) ---")
# Rang de volatilité relative
vol_ewm60_df = pd.DataFrame({pair: all_features_df[f'{pair}_log_range'].ewm(span=60, adjust=False).mean() for pair in PAIRS})
ranks = vol_ewm60_df.rank(axis=1, method='first')
scaled_ranks = 2 * ((ranks - 1) / (N_PAIRS - 1)) - 1

for pair in PAIRS:
    last_price = FX_df[f'FX_{pair}_last']
    # Volatilité journalière de fond
    all_features_df[f'{pair}_vol_journaliere_moyenne'] = all_features_df[f'{pair}_log_range'].ewm(span=1440, adjust=False).mean()
    
    # Position dans le range journalier
    min_1440, max_1440 = last_price.rolling(1440).min(), last_price.rolling(1440).max()
    all_features_df[f'{pair}_position_range_journalier'] = 2 * ((last_price - min_1440) / (max_1440 - min_1440 + EPSILON)) - 1
    
    # Rangs
    all_features_df[f'{pair}_vol_relative_rank'] = scaled_ranks[pair]
    all_features_df[f'{pair}_vol_relative_rank_change'] = scaled_ranks[pair].diff(60)

del vol_ewm60_df, ranks, scaled_ranks, FX_df
gc.collect()

--- Calcul des Features Cross-Section (Groupe 6) ---


0

In [5]:
print("--- Calcul des Features Temporelles (Groupe 7) ---")
timestamp = all_features_df.index
time_utc = timestamp.time

# Cycliques Intraday
all_features_df['minute_of_day_sin'] = np.sin(2 * np.pi * (timestamp.hour * 60 + timestamp.minute) / 1440)
all_features_df['minute_of_day_cos'] = np.cos(2 * np.pi * (timestamp.hour * 60 + timestamp.minute) / 1440)
all_features_df['day_of_week_sin'] = np.sin(2 * np.pi * timestamp.dayofweek / 5)
all_features_df['day_of_week_cos'] = np.cos(2 * np.pi * timestamp.dayofweek / 5)

# Session Flags
all_features_df['Is_Asia_Session'] = np.where(((time_utc >= dt.time(0, 0)) & (time_utc < dt.time(7, 0))), 1, -1)
all_features_df['Is_London_Premarket'] = np.where(((time_utc >= dt.time(7, 0)) & (time_utc < dt.time(7, 45))), 1, -1)
all_features_df['Is_London_Open'] = np.where(((time_utc >= dt.time(7, 45)) & (time_utc < dt.time(8, 15))), 1, -1)
all_features_df['Is_NY_Premarket'] = np.where(((time_utc >= dt.time(12, 0)) & (time_utc < dt.time(12, 45))), 1, -1)
all_features_df['Is_NY_Open'] = np.where(((time_utc >= dt.time(12, 45)) & (time_utc < dt.time(13, 15))), 1, -1)
all_features_df['Is_London_NY_Overlap'] = np.where(((time_utc >= dt.time(13, 15)) & (time_utc < dt.time(15, 45))), 1, -1)
all_features_df['Is_WMR_Fix'] = np.where(((time_utc >= dt.time(15, 45)) & (time_utc < dt.time(16, 15))), 1, -1)
all_features_df['Is_NY_Close'] = np.where(((time_utc >= dt.time(20, 45)) & (time_utc < dt.time(21, 15))), 1, -1)

print("Toutes les features calculées.")

--- Calcul des Features Temporelles (Groupe 7) ---
Toutes les features calculées.


In [6]:
print("--- Nettoyage et Sous-échantillonnage ---")
# 1. Suppression des NaN initiaux (dus aux rolling windows)
all_features_df.dropna(inplace=True)

# 2. Sous-échantillonnage à 15 minutes
features_15m = all_features_df[all_features_df.index.minute % PREDICTION_FREQUENCY == 0].copy()

# 3. Réorganisation : Cibles d'abord
target_cols = [f'{pair}_vol_cible_bps' for pair in PAIRS]
feature_cols = [col for col in features_15m.columns if col not in target_cols]
final_order = target_cols + sorted(feature_cols)
features_15m = features_15m[final_order]

print(f"Shape finale (15m) : {features_15m.shape}")

--- Nettoyage et Sous-échantillonnage ---
Shape finale (15m) : (724768, 144)


In [7]:
print("--- Sauvegarde ---")
output_file = OUTPUT_DIR / 'LSTMt_features_15m.parquet'
features_15m.to_parquet(output_file)
print(f"Fichier sauvegardé : {output_file}")

# Optionnel : Sauvegarde d'un fichier JSON listant les colonnes à normaliser pour usage ultérieur
import json

no_scale_suffixes = tuple(['mfi_centered', 'position_range_journalier', 'vol_relative_rank'])
time_features = ['minute_of_day_sin', 'minute_of_day_cos', 'day_of_week_sin', 'day_of_week_cos', 
                 'Is_Asia_Session', 'Is_London_Premarket', 'Is_London_Open', 'Is_NY_Premarket', 
                 'Is_NY_Open', 'Is_London_NY_Overlap', 'Is_WMR_Fix', 'Is_NY_Close']

scale_config = {
    "no_scale": [col for col in feature_cols if col in time_features or col.endswith(no_scale_suffixes)],
    "standard_scale": [col for col in feature_cols if not (col in time_features or col.endswith(no_scale_suffixes))],
    "targets": target_cols
}

with open(OUTPUT_DIR / 'feature_scaling_config.json', 'w') as f:
    json.dump(scale_config, f, indent=4)
print("Configuration de scaling sauvegardée.")

--- Sauvegarde ---
Fichier sauvegardé : parquets/LSTMt_features_15m.parquet
Configuration de scaling sauvegardée.
